# Optical tweezer arrays: Rydberg blockade

*A worked example for `rydberg.py` -- new tweezer-array physics, not an AMO.jl port.*

Two atoms in optical tweezers, both driven resonantly toward a Rydberg state. Whether both atoms
can be excited simultaneously depends entirely on their separation relative to the blockade radius
$r_b = (C_6/\Omega)^{1/6}$: inside it, the van der Waals shift `rydberg_interaction` computes pushes
the doubly-excited state so far off resonance that only one atom at a time can be excited -- the
actual mechanism behind neutral-atom two-qubit gates.

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent / "src"))

import numpy as np
import matplotlib.pyplot as plt

import htdse as ht
from htdse.submodules.spin import sigma_x

## 1. Blockade radius

`positions`/`C6` share whatever length unit you pick -- here an illustrative unit where the numbers
come out simple, not a real atomic species' actual C6 coefficient.

In [ ]:
C6 = 5.0 ** 6 * 1e6   # illustrative units
Omega = 1e6            # single-atom Rabi frequency, same units
r_b = ht.blockade_radius(C6, Omega)
print(f"blockade radius: {r_b:.2f}")

## 2. Two atoms, driven resonantly: inside vs. outside the blockade radius

Both atoms start in `ket("1")` (ground) and are driven with an independent `sigma_x` Rabi term;
`rydberg_interaction` adds the pairwise shift when both are in `ket("0")` (the Rydberg state).

In [ ]:
def two_atom_drive(separation, tmax, n=800):
    positions = np.array([[0.0, 0.0], [separation, 0.0]])
    H_int = ht.rydberg_interaction(positions, C6, state="0")
    with ht.quiet():
        H = (H_int + ht.term(0.5 * Omega * sigma_x, on="q0")
                   + ht.term(0.5 * Omega * sigma_x, on="q1"))
        psi0 = ht.otimes(ht.ket("1"), ht.ket("1"))
        ts = np.linspace(0, tmax, n)
        ev = ht.HamiltonianEvolution(H, psi0)
        pops = np.abs(ev.state_at(ts)) ** 2
    return ts, pops

tmax = 2 * np.pi / Omega * 1.2
fig, axes = plt.subplots(1, 2, figsize=(10, 4), sharey=True)
for ax, sep, label in zip(axes, (r_b / 3, r_b * 5),
                          ("well inside blockade radius", "far outside blockade radius")):
    ts, pops = two_atom_drive(sep, tmax)
    ax.plot(ts * Omega / (2 * np.pi), pops[:, 0], label="P(both excited)")
    ax.plot(ts * Omega / (2 * np.pi), pops[:, 1] + pops[:, 2], label="P(exactly one excited)")
    ax.set_title(f"{label}\n(r = {sep:.2f}, r_b = {r_b:.2f})")
    ax.set_xlabel("t / Rabi period")
    print(f"{label} (r={sep:.2f}): max P(both excited) = {pops[:,0].max():.4f}")
axes[0].set_ylabel("population")
axes[0].legend()
fig.suptitle("Rydberg blockade: two atoms driven resonantly toward the Rydberg state")
plt.show()

Inside the blockade radius the double-excitation channel is fully suppressed -- exactly one atom
gets excited, with the *pair* Rabi-flopping between "someone is excited" and "neither is." Outside
it, the atoms don't feel each other at all and both independently Rabi-flop all the way to fully
excited, exactly as two isolated qubits would.

## 3. Resonant dipole-dipole exchange (Foerster resonance)

`dipole_dipole_interaction` is the *resonant* counterpart to the off-resonant van der Waals shift
above -- a clean flip-flop that fully swaps an excitation between two atoms in different
dipole-coupled Rydberg states, with no drive needed at all.

In [ ]:
C3 = 50.0
positions = np.array([[0.0, 0.0], [4.0, 0.0]])
H_dd = ht.dipole_dipole_interaction(positions, C3)
coupling = C3 / 4.0 ** 3

with ht.quiet():
    psi0 = ht.otimes(ht.ket("0"), ht.ket("1"))  # atom 0 excited, atom 1 in ground
    ts = np.linspace(0, 2 * np.pi / coupling * 1.2, 400)
    ev = ht.HamiltonianEvolution(H_dd, psi0)
    pops = np.abs(ev.state_at(ts)) ** 2

fig, ax = plt.subplots(figsize=(5, 3))
ax.plot(ts, pops[:, 1], label="P(|01>)")
ax.plot(ts, pops[:, 2], label="P(|10>)")
ax.set_xlabel("t"); ax.set_ylabel("population")
ax.set_title("Resonant dipole-dipole excitation exchange")
ax.legend()
plt.show()